# Power BI Serving Layer

This notebook prepares governed, reporting-friendly tables for Power BI from the validated Gold dimensional model.

The completed analytics fact is restored below so this notebook is self-contained. The current lesson then creates only `dim_date_analytics_df`.

In [1]:
from pathlib import Path
import sys

import pandas as pd

current_directory = Path.cwd().resolve()

if (current_directory / "src").exists():
    PROJECT_ROOT = current_directory
elif (current_directory.parent / "src").exists():
    PROJECT_ROOT = current_directory.parent
else:
    raise FileNotFoundError(
        "Could not locate the Enterprise Banking Analytics Platform project root."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from configs.config import ANALYTICS_PATH, GOLD_PATH

dim_date_path = GOLD_PATH / "dim_date.parquet"
dim_merchant_path = GOLD_PATH / "dim_merchant.parquet"
fact_transaction_path = GOLD_PATH / "fact_transaction.parquet"

dim_date_df = pd.read_parquet(dim_date_path)
dim_merchant_df = pd.read_parquet(dim_merchant_path)
fact_transaction_df = pd.read_parquet(fact_transaction_path)

print("Gold tables loaded successfully.")

Gold tables loaded successfully.


## Completed prerequisite — analytics transaction fact

This is the already-completed fact-table contract from the previous lesson. It is kept here because the original notebook file was empty on disk.

In [2]:
fact_transaction_analytics_columns = [
    "transaction_key",
    "date_key",
    "merchant_key",
    "card_id",
    "transaction_timestamp",
    "transaction_amount",
    "transaction_method",
    "transaction_error",
    "is_fraud",
]

fact_transaction_analytics_df = (
    fact_transaction_df[
        fact_transaction_analytics_columns
    ]
    .copy()
)

fact_transaction_analytics_df["has_error"] = (
    fact_transaction_analytics_df[
        "transaction_error"
    ]
    .ne("NO_ERROR")
)

assert len(fact_transaction_analytics_df) == len(fact_transaction_df)
assert fact_transaction_analytics_df["transaction_key"].notna().all()
assert fact_transaction_analytics_df["transaction_key"].is_unique
assert fact_transaction_analytics_df["has_error"].sum() == 574

print("Analytics fact shape:", fact_transaction_analytics_df.shape)
print("Analytics fact validation passed.")
print()
print(fact_transaction_analytics_df["has_error"].value_counts())

Analytics fact shape: (19963, 10)
Analytics fact validation passed.

has_error
False    19389
True       574
Name: count, dtype: int64[pyarrow]


## Step 4A — Create the Power BI date dimension

The Gold date dimension is reusable across the platform. For Power BI, we now define an explicit reporting contract by copying only the fields report users need.

Two reporting fields are added:

- `calendar_year_month` gives charts an unambiguous month label such as `2002-12`.
- `calendar_year_month_sort` gives Power BI a numeric chronological order such as `200212`, preventing labels from being sorted alphabetically.

In [3]:
dim_date_analytics_columns = [
    "date_key",
    "full_date",
    "calendar_year",
    "calendar_quarter",
    "calendar_month_number",
    "calendar_month_name",
    "calendar_day_of_month",
    "calendar_day_of_week_number",
    "calendar_day_name",
    "is_weekend",
]

dim_date_analytics_df = (
    dim_date_df[
        dim_date_analytics_columns
    ]
    .copy()
)

dim_date_analytics_df["calendar_year_month"] = (
    dim_date_analytics_df["full_date"]
    .dt.strftime("%Y-%m")
)

dim_date_analytics_df["calendar_year_month_sort"] = (
    dim_date_analytics_df["calendar_year"] * 100
    + dim_date_analytics_df["calendar_month_number"]
)

In [4]:
print("Analytics date dimension shape:", dim_date_analytics_df.shape)

print("\nFirst five rows:")
print(dim_date_analytics_df.head().to_string(index=False))

print("\nData types:")
print(dim_date_analytics_df.dtypes.to_string())

print("\nYear-month boundary sample:")
print(
    dim_date_analytics_df.loc[
        dim_date_analytics_df["calendar_year_month"].isin(
            ["2002-12", "2003-01"]
        ),
        [
            "calendar_year_month",
            "calendar_year_month_sort",
        ],
    ]
    .drop_duplicates()
    .to_string(index=False)
)

Analytics date dimension shape: (6390, 12)

First five rows:
 date_key  full_date  calendar_year  calendar_quarter  calendar_month_number calendar_month_name  calendar_day_of_month  calendar_day_of_week_number calendar_day_name  is_weekend calendar_year_month  calendar_year_month_sort
 20020901 2002-09-01           2002                 3                      9           September                      1                            7            Sunday        True             2002-09                    200209
 20020902 2002-09-02           2002                 3                      9           September                      2                            1            Monday       False             2002-09                    200209
 20020903 2002-09-03           2002                 3                      9           September                      3                            2           Tuesday       False             2002-09                    200209
 20020904 2002-09-04           2002    

## Step 5 — Create the Power BI merchant dimension

This step creates a reporting-safe copy of the Gold merchant dimension using only the fields needed for merchant analysis and the relationship to the transaction fact.

Missing state and ZIP values are preserved. They can legitimately occur for online merchants or international locations, so replacing them with invented values would reduce data quality.

In [5]:
dim_merchant_analytics_columns = [
    "merchant_key",
    "merchant_id",
    "merchant_city",
    "merchant_state",
    "merchant_zip_code",
    "merchant_category_code",
]

dim_merchant_analytics_df = (
    dim_merchant_df[
        dim_merchant_analytics_columns
    ]
    .copy()
)

In [6]:
print("Analytics merchant dimension shape:", dim_merchant_analytics_df.shape)

print("\nFirst five rows:")
print(dim_merchant_analytics_df.head().to_string(index=False))

print("\nData types:")
print(dim_merchant_analytics_df.dtypes.to_string())

print("\nMissing values:")
print(dim_merchant_analytics_df.isna().sum().to_string())

Analytics merchant dimension shape: (1106, 6)

First five rows:
 merchant_key          merchant_id    merchant_city     merchant_state merchant_zip_code  merchant_category_code
            1 -9179793182211330025    Santo Domingo Dominican Republic              <NA>                    5812
            2 -9092677072201095172           ONLINE               <NA>              <NA>                    4900
            3 -8997856093426647374           Morton                 TX             79346                    5812
            4 -8992471532581037186 Huntington Beach                 CA             92646                    5411
            5 -8978688709093014088         La Verne                 CA             91750                    8021

Data types:
merchant_key               int64
merchant_id                Int64
merchant_city             string
merchant_state            string
merchant_zip_code         string
merchant_category_code     Int64

Missing values:
merchant_key                0


## Step 6A — Validate the Power BI date dimension

A valid date dimension must preserve every Gold date and have a complete, unique `date_key`. These conditions allow Power BI to use it safely as the one side of the date-to-transaction relationship.

The cell prints the measured counts before enforcing the requirements with assertions.

In [7]:
gold_date_row_count = len(dim_date_df)
analytics_date_row_count = len(dim_date_analytics_df)
date_key_null_count = (
    dim_date_analytics_df["date_key"]
    .isna()
    .sum()
)
date_key_duplicate_count = (
    dim_date_analytics_df["date_key"]
    .duplicated()
    .sum()
)

print("Gold date rows:", gold_date_row_count)
print("Analytics date rows:", analytics_date_row_count)
print("Missing analytics date keys:", date_key_null_count)
print("Duplicate analytics date keys:", date_key_duplicate_count)

assert analytics_date_row_count == gold_date_row_count
assert date_key_null_count == 0
assert date_key_duplicate_count == 0

print("Analytics date dimension validation passed.")

Gold date rows: 6390
Analytics date rows: 6390
Missing analytics date keys: 0
Duplicate analytics date keys: 0
Analytics date dimension validation passed.


## Step 6B — Validate the Power BI merchant dimension

A valid merchant dimension must preserve every Gold merchant and have a complete, unique `merchant_key`. The merchant key—not the descriptive location fields—is the primary key used on the one side of the merchant-to-transaction relationship.

Missing state and ZIP values remain allowed because they can be legitimate business data.

In [8]:
gold_merchant_row_count = len(dim_merchant_df)
analytics_merchant_row_count = len(dim_merchant_analytics_df)
merchant_key_null_count = (
    dim_merchant_analytics_df["merchant_key"]
    .isna()
    .sum()
)
merchant_key_duplicate_count = (
    dim_merchant_analytics_df["merchant_key"]
    .duplicated()
    .sum()
)

print("Gold merchant rows:", gold_merchant_row_count)
print("Analytics merchant rows:", analytics_merchant_row_count)
print("Missing analytics merchant keys:", merchant_key_null_count)
print("Duplicate analytics merchant keys:", merchant_key_duplicate_count)

assert analytics_merchant_row_count == gold_merchant_row_count
assert merchant_key_null_count == 0
assert merchant_key_duplicate_count == 0

print("Analytics merchant dimension validation passed.")

Gold merchant rows: 1106
Analytics merchant rows: 1106
Missing analytics merchant keys: 0
Duplicate analytics merchant keys: 0
Analytics merchant dimension validation passed.


## Step 6C — Validate the fact-to-date relationship

Every transaction `date_key` must match a `date_key` in the analytics date dimension. An unmatched value would create an orphaned fact row that Power BI could not assign to a valid calendar date.

The validation uses `isin()` to test membership, `~` to select the keys that do not match, and `sum()` to count those invalid foreign keys.

In [9]:
invalid_bi_date_fk_count = (
    ~fact_transaction_analytics_df["date_key"]
    .isin(dim_date_analytics_df["date_key"])
).sum()

print("Analytics fact rows checked:", len(fact_transaction_analytics_df))
print("Invalid analytics date foreign keys:", invalid_bi_date_fk_count)

assert invalid_bi_date_fk_count == 0

print("Fact-to-date relationship validation passed.")

Analytics fact rows checked: 19963
Invalid analytics date foreign keys: 0
Fact-to-date relationship validation passed.


## Step 6D — Validate the fact-to-merchant relationship

Every transaction `merchant_key` must match a `merchant_key` in the analytics merchant dimension. An unmatched value would create an orphaned fact row that Power BI could not attribute to a valid merchant.

The validation counts unmatched merchant foreign keys and requires the result to be zero.

In [10]:
invalid_bi_merchant_fk_count = (
    ~fact_transaction_analytics_df["merchant_key"]
    .isin(dim_merchant_analytics_df["merchant_key"])
).sum()

print("Analytics fact rows checked:", len(fact_transaction_analytics_df))
print("Invalid analytics merchant foreign keys:", invalid_bi_merchant_fk_count)

assert invalid_bi_merchant_fk_count == 0

print("Fact-to-merchant relationship validation passed.")

Analytics fact rows checked: 19963
Invalid analytics merchant foreign keys: 0
Fact-to-merchant relationship validation passed.


## Step 7A — Define the Power BI serving-file paths

The validated dataframes are currently in memory. Before writing them, this step defines stable destinations inside the configured analytics directory.

Using named path variables makes the serving contract explicit and prevents filenames from being repeated or accidentally written to inconsistent locations. No dataframe is written in this step.

In [11]:
ANALYTICS_PATH.mkdir(
    parents=True,
    exist_ok=True,
)

dim_date_analytics_path = (
    ANALYTICS_PATH
    / "dim_date_analytics.parquet"
)
dim_merchant_analytics_path = (
    ANALYTICS_PATH
    / "dim_merchant_analytics.parquet"
)
fact_transaction_analytics_path = (
    ANALYTICS_PATH
    / "fact_transaction_analytics.parquet"
)

print("Analytics directory:", ANALYTICS_PATH)
print("Analytics directory exists:", ANALYTICS_PATH.exists())
print("Date output:", dim_date_analytics_path)
print("Merchant output:", dim_merchant_analytics_path)
print("Transaction output:", fact_transaction_analytics_path)

Analytics directory: D:\Road Maps\project\Data Modeling\Enterprise-Banking-Analytics-Platform\data\analytics
Analytics directory exists: True
Date output: D:\Road Maps\project\Data Modeling\Enterprise-Banking-Analytics-Platform\data\analytics\dim_date_analytics.parquet
Merchant output: D:\Road Maps\project\Data Modeling\Enterprise-Banking-Analytics-Platform\data\analytics\dim_merchant_analytics.parquet
Transaction output: D:\Road Maps\project\Data Modeling\Enterprise-Banking-Analytics-Platform\data\analytics\fact_transaction_analytics.parquet


## Step 7B — Write the Power BI date-serving file

This step persists the validated `dim_date_analytics_df` dataframe as Parquet. Parquet preserves the table's column types and is efficient for analytical tools such as Power BI.

`index=False` prevents pandas' internal row index from becoming an unnecessary reporting column.

In [12]:
dim_date_analytics_df.to_parquet(
    dim_date_analytics_path,
    index=False,
)

print("Date serving file written:", dim_date_analytics_path)
print("Date serving file exists:", dim_date_analytics_path.exists())
print("Date serving file size (bytes):", dim_date_analytics_path.stat().st_size)

Date serving file written: D:\Road Maps\project\Data Modeling\Enterprise-Banking-Analytics-Platform\data\analytics\dim_date_analytics.parquet
Date serving file exists: True
Date serving file size (bytes): 103142


## Step 7C — Write the Power BI merchant-serving file

This step persists the validated `dim_merchant_analytics_df` dataframe as Parquet. The file preserves the approved merchant columns, their data types, and legitimate missing location values.

`index=False` prevents pandas' internal row index from becoming an unnecessary reporting column.

In [13]:
dim_merchant_analytics_df.to_parquet(
    dim_merchant_analytics_path,
    index=False,
)

print("Merchant serving file written:", dim_merchant_analytics_path)
print("Merchant serving file exists:", dim_merchant_analytics_path.exists())
print("Merchant serving file size (bytes):", dim_merchant_analytics_path.stat().st_size)

Merchant serving file written: D:\Road Maps\project\Data Modeling\Enterprise-Banking-Analytics-Platform\data\analytics\dim_merchant_analytics.parquet
Merchant serving file exists: True
Merchant serving file size (bytes): 25902


## Step 7D — Write the Power BI transaction-serving file

This step persists the validated `fact_transaction_analytics_df` dataframe as Parquet. Its grain remains one row per transaction, and the file includes the reporting-specific `has_error` flag.

`index=False` prevents pandas' internal row index from becoming an unnecessary reporting column.

In [14]:
fact_transaction_analytics_df.to_parquet(
    fact_transaction_analytics_path,
    index=False,
)

print("Transaction serving file written:", fact_transaction_analytics_path)
print("Transaction serving file exists:", fact_transaction_analytics_path.exists())
print("Transaction serving file size (bytes):", fact_transaction_analytics_path.stat().st_size)

Transaction serving file written: D:\Road Maps\project\Data Modeling\Enterprise-Banking-Analytics-Platform\data\analytics\fact_transaction_analytics.parquet
Transaction serving file exists: True
Transaction serving file size (bytes): 484458


## Step 8A — Read back and validate the date-serving file

This step reads the physical date Parquet file into a new persisted dataframe and compares it with the validated in-memory dataframe.

`pd.testing.assert_frame_equal()` verifies the complete table, including values, row order, column order, column names, and data types. The key checks are printed separately so the relationship requirements remain visible.

In [15]:
dim_date_analytics_persisted_df = pd.read_parquet(
    dim_date_analytics_path
)

date_columns_preserved = (
    dim_date_analytics_persisted_df.columns.tolist()
    == dim_date_analytics_df.columns.tolist()
)
persisted_date_key_null_count = (
    dim_date_analytics_persisted_df["date_key"]
    .isna()
    .sum()
)
persisted_date_key_duplicate_count = (
    dim_date_analytics_persisted_df["date_key"]
    .duplicated()
    .sum()
)

print("Persisted date shape:", dim_date_analytics_persisted_df.shape)
print("Date columns and order preserved:", date_columns_preserved)
print("Missing persisted date keys:", persisted_date_key_null_count)
print("Duplicate persisted date keys:", persisted_date_key_duplicate_count)

assert date_columns_preserved
assert persisted_date_key_null_count == 0
assert persisted_date_key_duplicate_count == 0
pd.testing.assert_frame_equal(
    dim_date_analytics_persisted_df,
    dim_date_analytics_df,
    check_dtype=True,
    check_like=False,
)

print("Date serving-file read-back validation passed.")

Persisted date shape: (6390, 12)
Date columns and order preserved: True
Missing persisted date keys: 0
Duplicate persisted date keys: 0
Date serving-file read-back validation passed.


## Step 8B — Read back and validate the merchant-serving file

This step reads the physical merchant Parquet file into a new persisted dataframe and compares it with the validated in-memory dataframe.

The merchant key must remain complete and unique. Legitimate missing state and ZIP values must also be preserved. `pd.testing.assert_frame_equal()` verifies the complete stored table.

In [16]:
dim_merchant_analytics_persisted_df = pd.read_parquet(
    dim_merchant_analytics_path
)

merchant_columns_preserved = (
    dim_merchant_analytics_persisted_df.columns.tolist()
    == dim_merchant_analytics_df.columns.tolist()
)
persisted_merchant_key_null_count = (
    dim_merchant_analytics_persisted_df["merchant_key"]
    .isna()
    .sum()
)
persisted_merchant_key_duplicate_count = (
    dim_merchant_analytics_persisted_df["merchant_key"]
    .duplicated()
    .sum()
)
persisted_merchant_state_null_count = (
    dim_merchant_analytics_persisted_df["merchant_state"]
    .isna()
    .sum()
)
persisted_merchant_zip_null_count = (
    dim_merchant_analytics_persisted_df["merchant_zip_code"]
    .isna()
    .sum()
)

print("Persisted merchant shape:", dim_merchant_analytics_persisted_df.shape)
print("Merchant columns and order preserved:", merchant_columns_preserved)
print("Missing persisted merchant keys:", persisted_merchant_key_null_count)
print("Duplicate persisted merchant keys:", persisted_merchant_key_duplicate_count)
print("Persisted missing merchant states:", persisted_merchant_state_null_count)
print("Persisted missing merchant ZIP codes:", persisted_merchant_zip_null_count)

assert merchant_columns_preserved
assert persisted_merchant_key_null_count == 0
assert persisted_merchant_key_duplicate_count == 0
pd.testing.assert_frame_equal(
    dim_merchant_analytics_persisted_df,
    dim_merchant_analytics_df,
    check_dtype=True,
    check_like=False,
)

print("Merchant serving-file read-back validation passed.")

Persisted merchant shape: (1106, 6)
Merchant columns and order preserved: True
Missing persisted merchant keys: 0
Duplicate persisted merchant keys: 0
Persisted missing merchant states: 57
Persisted missing merchant ZIP codes: 161
Merchant serving-file read-back validation passed.


## Step 8C — Read back and validate the transaction-serving file

This step reads the physical transaction Parquet file into a new persisted dataframe and compares it with the validated in-memory fact.

In addition to a complete dataframe comparison, the validation prints key integrity and business control totals for transaction amount, fraud, and errors.

In [17]:
fact_transaction_analytics_persisted_df = pd.read_parquet(
    fact_transaction_analytics_path
)

transaction_columns_preserved = (
    fact_transaction_analytics_persisted_df.columns.tolist()
    == fact_transaction_analytics_df.columns.tolist()
)
persisted_transaction_key_null_count = (
    fact_transaction_analytics_persisted_df["transaction_key"]
    .isna()
    .sum()
)
persisted_transaction_key_duplicate_count = (
    fact_transaction_analytics_persisted_df["transaction_key"]
    .duplicated()
    .sum()
)
persisted_transaction_amount_total = (
    fact_transaction_analytics_persisted_df["transaction_amount"]
    .sum()
)
persisted_fraud_transaction_count = (
    fact_transaction_analytics_persisted_df["is_fraud"]
    .sum()
)
persisted_error_transaction_count = (
    fact_transaction_analytics_persisted_df["has_error"]
    .sum()
)

print("Persisted transaction shape:", fact_transaction_analytics_persisted_df.shape)
print("Transaction columns and order preserved:", transaction_columns_preserved)
print("Missing persisted transaction keys:", persisted_transaction_key_null_count)
print("Duplicate persisted transaction keys:", persisted_transaction_key_duplicate_count)
print("Persisted transaction amount total:", persisted_transaction_amount_total)
print("Persisted fraudulent transactions:", persisted_fraud_transaction_count)
print("Persisted transactions with errors:", persisted_error_transaction_count)

assert transaction_columns_preserved
assert persisted_transaction_key_null_count == 0
assert persisted_transaction_key_duplicate_count == 0
assert persisted_error_transaction_count == 574
pd.testing.assert_frame_equal(
    fact_transaction_analytics_persisted_df,
    fact_transaction_analytics_df,
    check_dtype=True,
    check_like=False,
)

print("Transaction serving-file read-back validation passed.")

Persisted transaction shape: (19963, 10)
Transaction columns and order preserved: True
Missing persisted transaction keys: 0
Duplicate persisted transaction keys: 0
Persisted transaction amount total: 1622991.69
Persisted fraudulent transactions: 27
Persisted transactions with errors: 574
Transaction serving-file read-back validation passed.


## Step 8D — Validate the persisted Power BI star schema

This final boundary validation uses only the dataframes read back from the physical Parquet files. Every persisted transaction date and merchant foreign key must match its corresponding persisted dimension key.

Zero unmatched keys proves that the exact files supplied to Power BI retain valid one-to-many relationships.

In [18]:
invalid_persisted_date_fk_count = (
    ~fact_transaction_analytics_persisted_df["date_key"]
    .isin(dim_date_analytics_persisted_df["date_key"])
).sum()
invalid_persisted_merchant_fk_count = (
    ~fact_transaction_analytics_persisted_df["merchant_key"]
    .isin(dim_merchant_analytics_persisted_df["merchant_key"])
).sum()

print("Persisted fact rows checked:", len(fact_transaction_analytics_persisted_df))
print("Invalid persisted date foreign keys:", invalid_persisted_date_fk_count)
print("Invalid persisted merchant foreign keys:", invalid_persisted_merchant_fk_count)

assert invalid_persisted_date_fk_count == 0
assert invalid_persisted_merchant_fk_count == 0

print("Persisted Power BI star-schema validation passed.")

Persisted fact rows checked: 19963
Invalid persisted date foreign keys: 0
Invalid persisted merchant foreign keys: 0
Persisted Power BI star-schema validation passed.
